# Phase 4 — Fusion Model + Ablation Study
**Goal:** Build a model that uses BOTH price features AND FinBERT sentiment, then prove sentiment helps.

**What's an ablation study?**
We train 3 versions of the model and compare:
1. Price features only (no sentiment)
2. Sentiment features only (no price)
3. Price + Sentiment combined ← the full model

If version 3 beats versions 1 and 2, we've proven that combining NLP with time series adds real value.
This is standard practice in ML research papers.

## Step 1 — Setup

In [ ]:
!pip install torch pandas scikit-learn matplotlib numpy --quiet
print('Done!')

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Step 2 — Load data and define feature groups

In [ ]:
os.chdir('/content')
df = pd.read_csv('tsla_combined.csv')

# Create target: 1 if tomorrow's price > today's
df['next_close'] = df['Close'].shift(-1)
df['target'] = (df['next_close'] > df['Close']).astype(int)
df.dropna(subset=['next_close'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Three feature sets for ablation study
PRICE_FEATURES = [
    'daily_return', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist',
    'BB_upper', 'BB_mid', 'BB_lower', 'volume_change', 'MA_20', 'MA_50'
]
SENTIMENT_FEATURES = ['positive', 'negative', 'neutral', 'compound', 'n_headlines']
ALL_FEATURES = PRICE_FEATURES + SENTIMENT_FEATURES

print(f'Price features:     {len(PRICE_FEATURES)}')
print(f'Sentiment features: {len(SENTIMENT_FEATURES)}')
print(f'Combined features:  {len(ALL_FEATURES)}')
print(f'Dataset size: {len(df)} days')

## Step 3 — Train/val/test split and scaling

In [ ]:
WINDOW = 30
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

def scale_features(feature_cols, train_df, val_df, test_df):
    scaler = MinMaxScaler()
    train_s = scaler.fit_transform(train_df[feature_cols])
    val_s   = scaler.transform(val_df[feature_cols])
    test_s  = scaler.transform(test_df[feature_cols])
    return train_s, val_s, test_s, scaler

def create_sequences(features, labels, window):
    X, y = [], []
    for i in range(window, len(features)):
        X.append(features[i-window:i])
        y.append(labels[i])
    return np.array(X), np.array(y)

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

def prepare_loaders(feature_cols, batch_size=32):
    train_s, val_s, test_s, scaler = scale_features(feature_cols, train_df, val_df, test_df)
    labels_train = train_df['target'].values
    labels_val   = val_df['target'].values
    labels_test  = test_df['target'].values
    X_tr, y_tr = create_sequences(train_s, labels_train, WINDOW)
    X_va, y_va = create_sequences(val_s,   labels_val,   WINDOW)
    X_te, y_te = create_sequences(test_s,  labels_test,  WINDOW)
    tr_loader = DataLoader(StockDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(StockDataset(X_va, y_va), batch_size=batch_size, shuffle=False)
    te_loader = DataLoader(StockDataset(X_te, y_te), batch_size=batch_size, shuffle=False)
    return tr_loader, va_loader, te_loader

print('Helper functions defined!')

## Step 4 — Define LSTM model (same as Phase 3)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super(LSTMModel, self).__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, num_layers,
                               dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size, 32)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        out = self.dropout(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        return self.sigmoid(out).squeeze()

def train_model(input_size, train_loader, val_loader, epochs=50, lr=0.001):
    """Full training loop — returns trained model and history."""
    model     = LSTMModel(input_size=input_size).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5
    )
    best_val_loss    = float('inf')
    patience_counter = 0
    best_weights     = None
    history          = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        # Training
        model.train()
        batch_losses = []
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        # Validation
        model.eval()
        val_losses, val_preds, val_targets = [], [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                preds = model(X_b)
                val_losses.append(criterion(preds, y_b).item())
                val_preds.extend((preds > 0.5).cpu().numpy())
                val_targets.extend(y_b.cpu().numpy())

        val_loss = np.mean(val_losses)
        val_acc  = accuracy_score(val_targets, val_preds)
        scheduler.step(val_loss)

        history['train_loss'].append(np.mean(batch_losses))
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= 10:
            break

    model.load_state_dict(best_weights)
    return model, history

def evaluate(model, test_loader):
    """Evaluate model on test set, return metrics."""
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            preds = model(X_b.to(device))
            all_preds.extend((preds > 0.5).cpu().numpy())
            all_targets.extend(y_b.numpy().astype(int))
    acc = accuracy_score(all_targets, all_preds)
    f1  = f1_score(all_targets, all_preds)
    return acc, f1, all_preds, all_targets

print('Model and helper functions defined!')

## Step 5 — Ablation Study
Train all 3 model variants and compare.
This takes ~3-5 minutes total on GPU.

In [ ]:
ablation_results = {}

experiments = {
    'Price only':              PRICE_FEATURES,
    'Sentiment only':          SENTIMENT_FEATURES,
    'Price + Sentiment (Full)': ALL_FEATURES,
}

trained_models = {}

for name, features in experiments.items():
    print(f'\n── Training: {name} ({len(features)} features) ──')
    tr_loader, va_loader, te_loader = prepare_loaders(features)
    model, history = train_model(
        input_size=len(features),
        train_loader=tr_loader,
        val_loader=va_loader
    )
    acc, f1, preds, targets = evaluate(model, te_loader)
    ablation_results[name] = {'accuracy': acc, 'f1': f1, 'history': history}
    trained_models[name]   = model
    print(f'   Test Accuracy: {acc*100:.1f}%   F1: {f1:.4f}')

print('\n✓ All experiments complete!')

## Step 6 — Ablation results table

In [ ]:
print('\n' + '='*55)
print('           ABLATION STUDY RESULTS')
print('='*55)
print(f'{"Model":<30} {"Accuracy":>10} {"F1 Score":>10}')
print('-'*55)

for name, res in ablation_results.items():
    marker = ' ◄ BEST' if res['accuracy'] == max(r['accuracy'] for r in ablation_results.values()) else ''
    print(f'{name:<30} {res["accuracy"]*100:>9.1f}% {res["f1"]:>10.4f}{marker}')

print('='*55)

# Show improvement from adding sentiment
price_acc = ablation_results['Price only']['accuracy']
full_acc  = ablation_results['Price + Sentiment (Full)']['accuracy']
improvement = (full_acc - price_acc) * 100
print(f'\nSentiment contribution: {improvement:+.1f}% accuracy improvement')
print('(Full model vs Price only)')

## Step 7 — Visualise ablation comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Ablation Study — Impact of Sentiment Features', fontsize=13)

names  = list(ablation_results.keys())
accs   = [ablation_results[n]['accuracy'] * 100 for n in names]
f1s    = [ablation_results[n]['f1'] for n in names]
colors = ['#5B8DB8', '#E8A838', '#2ECC71']

# Accuracy bar chart
bars = axes[0].bar(names, accs, color=colors, edgecolor='white', width=0.5)
axes[0].axhline(50, color='red', linestyle='--', alpha=0.6, label='Random baseline (50%)')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Accuracy Comparison')
axes[0].set_ylim(45, max(accs) + 5)
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=15)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{acc:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# F1 bar chart
bars2 = axes[1].bar(names, f1s, color=colors, edgecolor='white', width=0.5)
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score Comparison')
axes[1].set_ylim(0, max(f1s) + 0.1)
axes[1].tick_params(axis='x', rotation=15)
for bar, f1 in zip(bars2, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{f1:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150)
plt.show()

## Step 8 — Training curves comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Validation Loss & Accuracy — All Models', fontsize=13)
colors_map = {
    'Price only':               '#5B8DB8',
    'Sentiment only':           '#E8A838',
    'Price + Sentiment (Full)': '#2ECC71'
}

for name, res in ablation_results.items():
    h = res['history']
    epochs = range(1, len(h['val_loss']) + 1)
    axes[0].plot(epochs, h['val_loss'], label=name, color=colors_map[name])
    axes[1].plot(epochs, h['val_acc'],  label=name, color=colors_map[name])

axes[0].set_title('Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_comparison.png', dpi=150)
plt.show()

## Step 9 — Save best model

In [ ]:
# Find and save the best performing model
best_name = max(ablation_results, key=lambda n: ablation_results[n]['accuracy'])
best_model = trained_models[best_name]

torch.save(best_model.state_dict(), 'best_fusion_model.pth')

# Save ablation results as CSV for your report
results_df = pd.DataFrame([
    {
        'model':    name,
        'accuracy': f"{res['accuracy']*100:.1f}%",
        'f1_score': f"{res['f1']:.4f}"
    }
    for name, res in ablation_results.items()
])
results_df.to_csv('ablation_results.csv', index=False)

print(f'Best model: {best_name}')
print(f'Accuracy:   {ablation_results[best_name]["accuracy"]*100:.1f}%')
print(f'F1 Score:   {ablation_results[best_name]["f1"]:.4f}')
print('\nSaved: best_fusion_model.pth, ablation_results.csv')
print('Saved: ablation_study.png, training_comparison.png')

## Summary

You now have a complete ablation study showing the value of combining FinBERT sentiment with LSTM price modeling.

**What to expect:**
- Price + Sentiment should beat Price only by 1-3%
- Even small improvements are meaningful in finance
- The comparison chart is your strongest result for the portfolio

**Next — Phase 5 (Final):** Build a simple Streamlit dashboard where you type a ticker and get a prediction. That's the deployment step that makes the project demo-able.